In [ ]:
# ========== 上传 .ipynb → 抽取代码 → 多后端加注释 + 摘要 ==========
# 导入 os：环境变量与密钥
import os
# 导入 json：解析 notebook JSON
import json
# Path：用 pathlib 读文件路径
from pathlib import Path
# load_dotenv：从 .env 加载 OPENAI_API_KEY / HF_TOKEN 等
from dotenv import load_dotenv
# Gradio：文件上传 + 文档化 UI
import gradio as gr
# 本地 Ollama Python 包（chat API）
import ollama
# OpenAI SDK：同时用于官方 API 与 HF Router 兼容接口
from openai import OpenAI


In [ ]:
# 加载 .env（默认不 override，与原调用一致）
load_dotenv()

# Hugging Face Token；缺省为空串
HF_TOKEN = os.getenv("HF_TOKEN", "")
# 把 OPENAI_API_KEY 写回环境，供后面 OpenAI() 默认构造读取
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

# 云端 GPT 模型 id
OPENAI_MODEL = "gpt-4o-mini"
# 本地 Llama 模型 id（Ollama 已拉取的名字）
LLAMA_MODEL = "llama3.1:8b"

# HF 免费聊天模型（经 Router）；字符串勿改
HF_FREE_CHAT_MODEL = "HuggingFaceTB/SmolLM3-3B:hf-inference"
# 备用模型行（注释保留，未启用）
#HF_FREE_CHAT_MODEL="katanemo/Arch-Router-1.5B"
# Hugging Face Router 的 OpenAI 兼容 Base URL
HF_ROUTER_BASE_URL = "https://router.huggingface.co/v1"

# 官方 OpenAI 客户端（读环境里的 OPENAI_API_KEY）
openai = OpenAI()
# HF Router 客户端：换 base_url，用 HF_TOKEN 作 api_key
hf_client = OpenAI(base_url=HF_ROUTER_BASE_URL, api_key=HF_TOKEN)


In [ ]:
# 从 .ipynb 路径抽出所有代码格源码，拼成一大段字符串
def notebook_code_extractor(path: str) -> str:
    # 按 UTF-8 读文件并 json.loads 成 dict
    nb = json.loads(Path(path).read_text(encoding="utf-8"))
    # parts：逐格收集代码源
    parts = []
    # 遍历 cells；只取 code 类型
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "code":
            # source 可能是字符串列表，join 成完整格源码
            parts.append("".join(cell.get("source", [])))
    # 格与格之间空一行；两端 strip
    return "\n\n".join(parts).strip()


In [ ]:
# system：请模型「只回复加了 docstring/注释的代码」（英文 prompt 勿改）
system_message_comments = (
    "You are a senior developer. Improve the code documentation by adding docstrings and short, useful comments. "
    "Keep it natural and practical. Do not over-comment obvious lines. "
    "Reply with code only."
)

# system：请模型用纯文本总结代码（不要代码、不要 Markdown）
system_message_summary = (
    "You are a senior developer. Summarize the code clearly: what it does, overall flow, inputs/outputs, and key points. "
    "Do not show the code. Do not use Markdown. Reply with plain text only."
)

# 拼「加注释」任务的 user 消息：固定英文指令 + 源码
def user_prompt_for(code: str) -> str:
    return "Add docstrings and helpful comments. Reply with code only.\n\n" + code

# 拼「摘要」任务的 user 消息
def user_prompt_for_summary(code: str) -> str:
    return "Summarize this code.\n\n" + code

# 组装 chat messages：system(注释规则) + user(代码)
def messages_for(code: str):
    return [
        {"role": "system", "content": system_message_comments},
        {"role": "user", "content": user_prompt_for(code)},
    ]

# 组装 chat messages：system(摘要规则) + user(代码)
def messages_for_summary(code: str):
    return [
        {"role": "system", "content": system_message_summary},
        {"role": "user", "content": user_prompt_for_summary(code)},
    ]


In [ ]:
# 三个后端封装：各自跑「加注释」+「摘要」两次调用，返回 (commented, summary)

def call_llama_local(code: str):
    # 本地 Ollama：dict 风格响应，内容在 ["message"]["content"]
    r1 = ollama.chat(model=LLAMA_MODEL, messages=messages_for(code))
    r2 = ollama.chat(model=LLAMA_MODEL, messages=messages_for_summary(code))
    return r1["message"]["content"], r2["message"]["content"]

def call_gpt(code: str):
    # 官方 OpenAI Chat Completions
    c1 = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(code))
    c2 = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for_summary(code))
    return c1.choices[0].message.content, c2.choices[0].message.content

def call_hf_free(code: str):
    # 无 HF_TOKEN 则直接报错（英文错误文案保留，供程序/用户识别）
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is not set in your environment.")
    # 经 HF Router 调免费聊天模型；限制 max_tokens
    c1 = hf_client.chat.completions.create(
        model=HF_FREE_CHAT_MODEL,
        messages=messages_for(code),
        max_tokens=1000,
    )
    c2 = hf_client.chat.completions.create(
        model=HF_FREE_CHAT_MODEL,
        messages=messages_for_summary(code),
        max_tokens=1000,
    )
    return c1.choices[0].message.content, c2.choices[0].message.content


In [ ]:
# Gradio 回调：上传笔记本 → 抽代码 → 按所选模型文档化
import traceback

def document_uploaded_notebook(file_obj, model: str):
    try:
        # 未上传文件
        if file_obj is None:
            return "ERROR: Please upload a .ipynb file.", "", ""

        # Gradio File 对象的本地临时路径
        path = file_obj.name
        # 扩展名校验：必须是 .ipynb
        if not path.lower().endswith(".ipynb"):
            return "ERROR: The uploaded file is not a .ipynb notebook.", "", ""

        # 抽出全部代码格
        code = notebook_code_extractor(path)
        if not code.strip():
            return "ERROR: No code cells found in the notebook.", "", ""

        # 下拉显示名转小写，用前缀分流
        m = (model or "").strip().lower()
        if m.startswith("llama"):
            # 本地 Llama
            commented, summary = call_llama_local(code)
        elif m.startswith("gpt"):
            # OpenAI GPT
            commented, summary = call_gpt(code)
        elif m.startswith("hf"):
            # Hugging Face 免费路由
            commented, summary = call_hf_free(code)
        else:
            # 未知选项
            return f"ERROR: Unsupported model: {model!r}", "", ""

        # 成功：原代码、带注释代码、摘要
        return code, commented, summary

    except Exception:
        # 任意异常：把 traceback 放进第一个输出框
        return "ERROR:\n" + traceback.format_exc(), "", ""


In [ ]:
# Gradio CSS：给「文档化代码」「摘要」两个框上不同背景色
css = """
.comments {background-color: #00599C;}
.summary {background-color: #008B8B;}
"""

# Blocks + 自定义 css
with gr.Blocks(css=css) as ui:
    # 标题与用法说明（UI 英文保留）
    gr.Markdown("### Notebook Documentation Tool\nUpload a notebook and generate docstrings/comments + a summary.")

    with gr.Row():
        # 只允许选 .ipynb
        nb_file = gr.File(label="Upload .ipynb", file_types=[".ipynb"])

    with gr.Row():
        # 三个后端选项；默认 HF (free)；前缀供 startswith 分流
        model = gr.Dropdown(
            ["HF (free)", "Llama (local)", "GPT (API)"],
            label="Model",
            value="HF (free)",
        )

    with gr.Row():
        # 触发 document_uploaded_notebook
        run = gr.Button("Generate documentation")

    with gr.Row():
        # 只读：展示从笔记本抽出的源码
        source_code = gr.Textbox(label="Extracted notebook code (read-only)", lines=14, interactive=False)

    with gr.Row():
        # 左：模型加注释后的代码；右：纯文本摘要
        commented_code = gr.Textbox(label="Documented code", lines=14, elem_classes=["comments"])
        code_summary = gr.Textbox(label="Summary", lines=14, elem_classes=["summary"])

    # 绑定：文件 + 模型 → 三路输出
    run.click(
        document_uploaded_notebook,
        inputs=[nb_file, model],
        outputs=[source_code, commented_code, code_summary],
    )

# 启动并尝试打开浏览器
ui.launch(inbrowser=True)
